In [1]:
from pathlib import Path
import os

from caf.base import DVector

os.chdir(r'..\..')
from land_use.norcom import apply_norcom, NorCOMResult
os.chdir(r'supporting_processes\norcom')

In [2]:
# Define path to store outputs
output_dir = Path(r"F:\Working\Land-Use\NorCOM 2023 Application")

In [3]:
# define path to household and population inputs from 2023 base year model
input_dir = Path(r"F:\Deliverables\Land-Use\241220_Populationv2\02_Final Outputs")

In [4]:
# define household and population files
result_files = {
    'household': list(input_dir.glob("Output P13.3_*.hdf")),
    'population': list(input_dir.glob("Output P11_*.hdf"))
}

In [5]:
# load the NorCOM results
any_car_ownership = NorCOMResult.from_coefficients_csv(
    csv_path=Path(r"I:\NorMITs NorCOM\AECOM working\v38\0v1+\output\final_model_coefficients.csv"),
    case_category='1+', noncase_category='0',
    zonal_lookups=Path(r"I:\NorMITs NorCOM\Import\Zonal Data\zonal_logit_data_v38_2023.csv")
)

multiple_car_ownership = NorCOMResult.from_coefficients_csv(
    csv_path=Path(r"I:\NorMITs NorCOM\AECOM working\v38\1v2+\output\final_model_coefficients.csv"),
    case_category='2+', noncase_category='1', dependent_category='1+',
    zonal_lookups=Path(r"I:\NorMITs NorCOM\Import\Zonal Data\zonal_logit_data_v38_2023.csv")
)

C:\Miniforge3\envs\normits_lu\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Miniforge3\envs\normits_lu\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [6]:
for file in result_files["household"]:
    # skip scotland for now
    if file.with_suffix("").name.endswith("Scotland"):
        continue
        
    # read in 2023 household data
    input_households = DVector.load(file)

    # get file name
    ref = file.name

    # get region from file name to get corresponding population file
    gor = file.with_suffix("").name.split("_")[-1]
    
    # apply norcom
    output_households = apply_norcom(
        any_car_ownership_result=any_car_ownership, 
        multiple_car_ownership_result=multiple_car_ownership,
        input_dvector=input_households
    )

    # save output households
    output_households.save(output_dir / ref)

    # read in population (pre-norcom)
    input_population = DVector.load(file.parent / f"Output P11_{gor}.hdf")

    # get segmentations from the resulting households *except* car availability
    hh_segs = [
        seg for seg in output_households.segmentation.names if seg != "car_availability"
    ]

    # calculate factors of car available splits by zone and segment that have resulted from norcom application
    factors = output_households / output_households.aggregate(hh_segs)

    # get segmentations from the input population *except* car availability
    pop_segs = [
        seg for seg in input_population.segmentation.names if seg != "car_availability"
    ]

    # apply factors to population
    output_pop = input_population.aggregate(pop_segs) * factors

    # save output households
    output_pop.save(output_dir / f"Output P11_{gor}.hdf")

The input data started with 35,672 columns. Filtering to None results in 2,847 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:931: SegmentationWarning: This operation has changed the segmentation of the DVector from ['children', 'adults', 'ns_sec', 'adult_nssec', 'accom_h', 'total'] to ['ns_sec', 'adults', 'total', 'adult_nssec', 'accom_h', 'children', 'car_availability']. This can happen but it can also be a sign of an error. Check the output DVector.
  warnings.warn(
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\segmentation.py:342: SegmentationWarning: Read in level car_availability is a subset of the segment. If this was not expected check the input segmentation.
  warnings.warn(
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\segmenta